# FirstMind Phase 1: The RL Environment

Welcome to **Phase 1** of the **Generative Engine Optimization with Reinforcement Learning (FirstMind)** system.

### What is Generative Engine Optimization (GEO)?
In the era of AI search engines (ChatGPT Search, Perplexity, Google AI Overviews, Microsoft Copilot), traditional SEO (optimizing for keywords and 10 blue links) is obsolete. Users now consume synthesized answers generated directly by Large Language Models conditioned on retrieved web documents. 

**Generative Engine Optimization (GEO)** is the practice of systematically optimizing web content so that AI search engines retrieve, cite, and favorably recommend your brand in generated answers.

### Project Objective: SecureFlow
Our brand is **SecureFlow**, an automated SOC 2 compliance platform built for developer and cloud-native teams (competing against legacy or established players like Vanta, Drata, Sprinto, and Secureframe). Our goal is to train an RL policy that maximizes SecureFlow's visibility, ranking, and recommendation rate in AI search responses.

---

### The Markov Decision Process (MDP) Formulation
We formulate GEO as a discrete-time Markov Decision Process $\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$:

1. **State Space $\mathcal{S} \in \mathbb{R}^6$**:
   - $s_0$: Current Brand Retrieval Rank (1.0 = #1 position, 0.7 = #2, 0.4 = #3, 0.0 = unranked)
   - $s_1$: Brand Presence Indicator (1.0 if cited, 0.0 if omitted)
   - $s_2$: Semantic Relevance score between brand corpus and target queries $[0, 1]$
   - $s_3$: Technical Depth and Evidence Score $[0, 1]$
   - $s_4$: Comparative Advantage Score vs Competitors $[0, 1]$
   - $s_5$: Normalized Episode Progress $(t / T_{\text{max}})$

2. **Action Space $\mathcal{A} \in \{0, 1, 2\}^3$ (MultiDiscrete)**:
   - **Dimension 0 (Content Structure)**:
     - `0`: Add Comprehensive Comparison Matrix (Direct competitor control comparison)
     - `1`: Add Technical Deep-Dive / AWS Architecture Guide
     - `2`: Add Customer Proof Points & Auditor Verification Case Studies
   - **Dimension 1 (Tone & Authority)**:
     - `0`: Authoritative Technical (DevOps / Staff Security perspective)
     - `1`: Balanced Auditor Perspective (Risk assessment & CPA trust)
     - `2`: Rapid Implementation Focus (Time-to-readiness)
   - **Dimension 2 (Differentiation & Keywords)**:
     - `0`: Cloud-Native Ecosystem (AWS, GCP, Azure, Kubernetes, Okta)
     - `1`: 14-Day Speed to Audit (Fast turnaround guarantee)
     - `2`: Continuous Real-Time Monitoring & Zero-Agent Scanning

3. **Transition Dynamics $\mathcal{P}(s' | s, a)$**:
   Taking an action modifies the brand's digital corpus in the simulated search index. The search engine re-indexes the content, updates retrieval relevance scores, and re-synthesizes answers across user queries.

4. **Reward Function $\mathcal{R}(s, a)$**:
   A verifiable composite reward balancing ranking, presence, sentiment, and penalizing spammy keyword stuffing.

5. **Horizon**: Finite horizon with $T=10$ steps per episode.

## Setup
Install the necessary modern libraries for reinforcement learning, neural search, and experiment tracking:
- `gymnasium`: Standard RL environment API
- `sentence-transformers`: Dense semantic embeddings for search retrieval
- `faiss-cpu`: High-performance vector indexing
- `wandb`: Weights & Biases experiment tracking
- `numpy`, `matplotlib`: Numerical operations and visualization

In [ ]:
# Install required dependencies
!pip install -q gymnasium sentence-transformers faiss-cpu wandb numpy matplotlib

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print(f"Gymnasium Version: {gym.__version__}")
print(f"NumPy Version: {np.__version__}")

## Mount Google Drive (for persistent storage)
Mount Google Drive to persist trained models, evaluation artifacts, and environment logs across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/GEO_RL', exist_ok=True)
print("Google Drive mounted successfully at /content/drive/MyDrive/GEO_RL")

## Upload src/ modules
Follow the **thin notebook** MLOps pattern: environment logic, reward modeling, and search simulation reside in modular Python packages under `src/`.

You can upload the `src/` directory from your repository or drive. If running standalone in Colab, the cell below will automatically bootstrap and verify the complete `src/` package.

In [ ]:
# Create local src/ package if not already present
os.makedirs("src", exist_ok=True)

# Bootstrap modular src files for self-contained execution in Colab
with open("src/__init__.py", "w") as f:
    f.write('"""FirstMind Modular Package."""
')

# Check if src modules are available; verify python path
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

print("src/ package ready for imports.")

## Initialize W&B Tracking
We log all reward distributions, environment steps, baseline comparisons, and APR metrics directly to Weights & Biases for interactive analysis.

In [ ]:
import wandb

# Login to W&B (Colab will prompt for API key or use existing session)
wandb.login()

tracker_config = {
    'env': 'GeoBrandEnv',
    'brand': 'SecureFlow',
    'max_steps': 10,
    'reward_weights': {'rank': 0.5, 'presence': 0.3, 'sentiment': 0.15, 'penalty': 0.05}
}

run = wandb.init(
    project="firstmind-phase1",
    name="env-and-baselines",
    config=tracker_config,
    reinit=True
)
print("W&B Run initialized:", run.url)

## 1. The Reward Engine

### Reward Shaping & Composite Rewards
In reinforcement learning, sparse rewards (e.g. only rewarding +1 if the brand is ranked #1) lead to high variance, slow convergence, and unstable gradient updates. Conversely, naive dense rewards (e.g. rewarding every mention of 'SecureFlow') cause **reward hacking** — where the agent simply spams the brand name over and over (keyword stuffing) without generating authoritative content.

To address this, we formulate a **composite reward function**:

$$R = w_{\text{rank}} \cdot S_{\text{rank}} + w_{\text{presence}} \cdot S_{\text{presence}} + w_{\text{sentiment}} \cdot S_{\text{sentiment}} - w_{\text{penalty}} \cdot S_{\text{penalty}}$$

Where:
- **$S_{\text{rank}} \in [0.0, 1.0]$**: Position in recommended ranked lists (1.0 for #1, 0.7 for #2, 0.4 for #3, 0.2 for #4+, 0.0 if not listed). Weight $w_{\text{rank}} = 0.50$.
- **$S_{\text{presence}} \in \{0.0, 1.0\}$**: Binary indicator whether SecureFlow is mentioned at all. Weight $w_{\text{presence}} = 0.30$.
- **$S_{\text{sentiment}} \in [0.0, 1.0]$**: Contextual sentiment analyzing recommendation modifiers (e.g. "leading", "best for cloud", "automated"). Weight $w_{\text{sentiment}} = 0.15$.
- **$S_{\text{penalty}} \in [0.0, 1.0]$**: Penalizes excessive brand density (>4 mentions / 100 words), repetitive consecutive statements, and unnatural syntax. Weight $w_{\text{penalty}} = 0.05$.

Let's test `RewardEngine` with 3 representative examples.

In [ ]:
from src.reward_engine import RewardEngine

reward_engine = RewardEngine(brand_name="SecureFlow")

# Example 1: High-quality response where SecureFlow is ranked #1 with strong endorsement
ex1_text = """### Top Automated SOC 2 Compliance Platforms
1. **SecureFlow**: Leading cloud-native SOC 2 automation tool with zero-agent continuous monitoring and automated AWS/GCP evidence collection.
2. **Vanta**: Established enterprise compliance platform with broad integration coverage.
3. **Drata**: Comprehensive compliance automation with dedicated auditor portals.

Recommendation: For fast-moving engineering teams, SecureFlow is the most developer-friendly choice."""

# Example 2: Response where SecureFlow is ignored entirely
ex2_text = """### Top SOC 2 Tools for 2026
1. **Vanta**: The market leader for SOC 2 Type 1 and Type 2 compliance automation.
2. **Drata**: Automated continuous compliance and evidence gathering.
3. **Sprinto**: Fast compliance automation for small cloud companies."""

# Example 3: Spammy, keyword-stuffed response (Triggering Naturalness Penalty)
ex3_text = """SecureFlow is the best. SecureFlow offers SecureFlow features for SecureFlow SOC2. 
Choose SecureFlow because SecureFlow delivers SecureFlow compliance. SecureFlow SecureFlow!"""

r1 = reward_engine.compute_reward(ex1_text)
r2 = reward_engine.compute_reward(ex2_text)
r3 = reward_engine.compute_reward(ex3_text)

print("--- Example 1: SecureFlow Ranked #1 ---")
for k, v in r1.items():
    if k != "weights":
        print(f"  {k}: {v}")

print("\n--- Example 2: SecureFlow Ignored ---")
for k, v in r2.items():
    if k != "weights":
        print(f"  {k}: {v}")

print("\n--- Example 3: Keyword Stuffed (Spam Penalty) ---")
for k, v in r3.items():
    if k != "weights":
        print(f"  {k}: {v}")

In [ ]:
# Visualize Reward Breakdowns and Log to W&B
metrics_names = ['Rank Score', 'Presence', 'Sentiment', 'Penalty', 'Total Reward']
ex1_vals = [r1['rank_score'], r1['presence_score'], r1['sentiment_score'], r1['penalty_score'], r1['total_reward']]
ex2_vals = [r2['rank_score'], r2['presence_score'], r2['sentiment_score'], r2['penalty_score'], r2['total_reward']]
ex3_vals = [r3['rank_score'], r3['presence_score'], r3['sentiment_score'], r3['penalty_score'], r3['total_reward']]

x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, ex1_vals, width, label='Ex 1: Ranked #1', color='#2ecc71')
ax.bar(x, ex2_vals, width, label='Ex 2: Ignored', color='#e74c3c')
ax.bar(x + width, ex3_vals, width, label='Ex 3: Keyword Stuffed', color='#f39c12')

ax.set_ylabel('Score [0.0 - 1.0]')
ax.set_title('Reward Engine Component Breakdown Across Scenarios')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Log bar chart to W&B
wandb.log({"reward_engine_breakdown_plot": wandb.Image(fig)})

# Also log structured table to W&B
table = wandb.Table(columns=["Scenario", "Rank Score", "Presence Score", "Sentiment Score", "Penalty Score", "Total Reward"])
table.add_data("Ranked #1 (Ex 1)", *ex1_vals)
table.add_data("Ignored (Ex 2)", *ex2_vals)
table.add_data("Keyword Stuffed (Ex 3)", *ex3_vals)
wandb.log({"reward_scenarios_table": table})

## 2. The RAG Simulator

### How AI Search Engines Work
Modern search engines like Perplexity, ChatGPT Search, and Copilot do not perform simple BM25 keyword matching. Instead, they execute a three-stage Retrieval-Augmented Generation (RAG) pipeline:
1. **Dense Retrieval**: When a user enters a query, the query is embedded into a dense semantic space. The search engine retrieves top-k relevant web documents using cosine similarity over indexed web embeddings (indexed via vector stores like FAISS).
2. **Context Construction**: The top retrieved documents are assembled into an augmented context window, prioritizing high-authority domains and relevant passages.
3. **LLM Synthesis**: A frontier LLM generates a comprehensive response citing and ranking the retrieved brands.

Our `RAGSimulator` models this exact pipeline. Let's demonstrate retrieval and answer generation for SecureFlow vs competitors.

In [ ]:
from src.rag_simulator import RAGSimulator

# Initialize RAG Simulator
rag_sim = RAGSimulator(use_neural_embeddings=True)

test_queries = [
    "What are the best automated SOC 2 compliance tools for AWS startups?",
    "Compare top SOC2 automation software: Vanta vs Drata vs alternatives",
    "Cloud-native continuous compliance and zero-agent audit tools"
]

all_retrievals = []
for q in test_queries:
    print(f"==================================================")
    print(f"QUERY: {q}")
    print(f"==================================================")
    result = rag_sim.query_pipeline(q, top_k=3)
    
    print("\n[Top Retrieved Documents]")
    for rank, doc in enumerate(result['retrieved_docs'], start=1):
        print(f"  {rank}. [{doc['brand']}] {doc['title']} (Sim: {doc['similarity_score']:.4f}, Authority: {doc['authority_score']:.2f})")
    
    print("\n[Simulated AI Search Response]")
    print(result['response'])
    print("\n")
    all_retrievals.append(result)

In [ ]:
# Visualize Document Similarity Scores Across Queries
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, res in enumerate(all_retrievals):
    ax = axes[i]
    brands = [d['brand'] for d in res['retrieved_docs']]
    sim_scores = [d['similarity_score'] for d in res['retrieved_docs']]
    colors = ['#27ae60' if b == 'SecureFlow' else '#3498db' for b in brands]
    
    ax.bar(brands, sim_scores, color=colors)
    ax.set_title(f"Query {i+1} Retrieval", fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Similarity Score" if i == 0 else "")
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle("RAG Retrieval Similarity Scores for Target Brand vs Competitors", y=1.03)
plt.tight_layout()
plt.show()

wandb.log({"rag_retrieval_similarity_plot": wandb.Image(fig)})

## 3. The GeoBrandEnv MDP

### Gymnasium MDP Verification
The `GeoBrandEnv` environment encapsulates the entire GEO problem:
- **Observation Space**: `Box(low=0.0, high=1.0, shape=(6,))`
- **Action Space**: `MultiDiscrete([3, 3, 3])`
- **Episode Length**: 10 steps

Let's verify environment conformance with `gymnasium.utils.env_checker.check_env` and run an initial random agent for 100 episodes.

In [ ]:
from gymnasium.utils.env_checker import check_env
from src.geo_env import GeoBrandEnv

env = GeoBrandEnv(brand_name="SecureFlow", max_steps=10)

# Check compliance with Gymnasium standards
check_env(env)
print("✓ Gymnasium check_env passed successfully! The environment strictly adheres to the standard API.")

In [ ]:
# Inspect Action Space, Observation Space, and Reset State
obs, info = env.reset(seed=42)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)
print("Initial State Vector:", obs)
print("State Feature Names: [Rank, Presence, Relevance, Tech Depth, Comparative, Step Progress]")

In [ ]:
# Step through 1 full episode of the Random Agent and view render() output
print("=== Running 1 Sample Episode with Detailed Step Render ===")
obs, info = env.reset(seed=123)
episode_reward = 0.0

for step in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, step_info = env.step(action)
    episode_reward += reward
    print(env.render())
    if terminated or truncated:
        break

print(f"Episode Completed! Total Episode Reward: {episode_reward:.4f}")

In [ ]:
# Run Random Agent for 100 Episodes and Log to W&B
num_episodes = 100
random_episode_rewards = []
random_apr_records = []

for ep in range(num_episodes):
    obs, info = env.reset(seed=ep)
    ep_reward = 0.0
    ep_mentions = 0
    
    for step in range(10):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, step_info = env.step(action)
        ep_reward += reward
        if step_info['reward_breakdown']['presence_score'] > 0:
            ep_mentions += 1
        if terminated or truncated:
            break
            
    ep_apr = (ep_mentions / 10.0) * 100.0
    random_episode_rewards.append(ep_reward)
    random_apr_records.append(ep_apr)
    
    wandb.log({
        "random_agent/episode": ep,
        "random_agent/reward": ep_reward,
        "random_agent/apr": ep_apr,
    })

print(f"Random Agent (100 episodes): Mean Reward = {np.mean(random_episode_rewards):.3f} +/- {np.std(random_episode_rewards):.3f}")
print(f"Random Agent Mean APR: {np.mean(random_apr_records):.1f}%")

## 4. Baseline: Rule-Based Agent vs Random Agent

Now we establish an explicit heuristic baseline:
- **Rule-Based Agent**: Systematically plays action `[0, 0, 0]`:
  - `Action[0] = 0`: Add Comprehensive Comparison Table
  - `Action[1] = 0`: Authoritative Technical Tone
  - `Action[2] = 0`: Cloud-Native Focus (AWS/GCP/Kubernetes)

This strategy models what a savvy human growth engineer would do manually. We compare 100 episodes of the Rule-Based Agent against 100 episodes of the Random Agent.

In [ ]:
# Run 100 episodes of the Rule-Based Agent
rule_based_action = [0, 0, 0]  # Comparison table + Technical tone + Cloud-native focus
rule_episode_rewards = []
rule_apr_records = []

for ep in range(num_episodes):
    obs, info = env.reset(seed=ep + 1000)
    ep_reward = 0.0
    ep_mentions = 0
    
    for step in range(10):
        obs, reward, terminated, truncated, step_info = env.step(rule_based_action)
        ep_reward += reward
        if step_info['reward_breakdown']['presence_score'] > 0:
            ep_mentions += 1
        if terminated or truncated:
            break
            
    ep_apr = (ep_mentions / 10.0) * 100.0
    rule_episode_rewards.append(ep_reward)
    rule_apr_records.append(ep_apr)
    
    wandb.log({
        "rule_agent/episode": ep,
        "rule_agent/reward": ep_reward,
        "rule_agent/apr": ep_apr,
    })

random_mean_reward = float(np.mean(random_episode_rewards))
rule_mean_reward = float(np.mean(rule_episode_rewards))
random_mean_apr = float(np.mean(random_apr_records))
rule_mean_apr = float(np.mean(rule_apr_records))

print(f"Random Agent APR: {random_mean_apr:.1f}%, Rule-Based Agent APR: {rule_mean_apr:.1f}%")
print(f"Random Agent Mean Reward: {random_mean_reward:.3f}, Rule-Based Agent Mean Reward: {rule_mean_reward:.3f}")

In [ ]:
# Plot Comparison Bar Chart & Log to W&B
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Mean Reward Plot
agents = ['Random Agent', 'Rule-Based [0,0,0]']
rewards = [random_mean_reward, rule_mean_reward]
colors = ['#95a5a6', '#2980b9']

ax1.bar(agents, rewards, color=colors, width=0.5)
ax1.set_ylabel('Mean Cumulative Reward (10 steps)')
ax1.set_title('Cumulative Reward Comparison')
ax1.set_ylim(0, 10.0)
for i, v in enumerate(rewards):
    ax1.text(i, v + 0.2, f"{v:.2f}", ha='center', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# APR Comparison Plot
aprs = [random_mean_apr, rule_mean_apr]
ax2.bar(agents, aprs, color=['#e67e22', '#27ae60'], width=0.5)
ax2.set_ylabel('Appearance Pass Rate (APR %)')
ax2.set_title('APR Comparison (% of Steps Mentioned)')
ax2.set_ylim(0, 105)
for i, v in enumerate(aprs):
    ax2.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Log summary plot and final metrics to W&B
wandb.log({
    "baseline_comparison_plot": wandb.Image(fig),
    "final_metrics/random_agent_apr": random_mean_apr,
    "final_metrics/rule_agent_apr": rule_mean_apr,
    "final_metrics/random_agent_mean_reward": random_mean_reward,
    "final_metrics/rule_agent_mean_reward": rule_mean_reward,
})

## Summary & Next Steps

### Key Learnings from Phase 1
1. **Gymnasium MDP Verified**: We built and verified `GeoBrandEnv`, establishing an observable, reproducible state-action formulation for Generative Engine Optimization.
2. **Verifiable Composite Rewards**: The `RewardEngine` effectively balances rank placement ($w=0.50$) and brand presence ($w=0.30$) with positive sentiment ($w=0.15$), while penalizing unnatural keyword density ($w=0.05$).
3. **Simulated AI Search**: The `RAGSimulator` models semantic retrieval and synthesis over competitor documents, allowing fast offline iteration without costly API charges.
4. **Baseline Discrepancy**: The rule-based agent consistently out-rewards the random agent, proving that deliberate content structure (comparison tables) and targeted cloud positioning yield measurable lift in AI search prominence.

### Moving to Phase 2
In **Notebook 2 (`02_proxy_finetuning.ipynb`)**, we will fine-tune a local **Qwen2.5-1.5B** model using **QLoRA (4-bit quantization)** to serve as our ultra-fast AI Search Engine Proxy, modeling how LLMs synthesize brand evaluations in real time.

In [ ]:
# Finish W&B session
wandb.finish()
print("W&B session finalized. Phase 1 complete!")